# Baseline Evaluation and Indexing

This notebook establishes the performance floor (Baseline) for the retrieval system. It builds the initial Sparse Indices (SPLADE and BM25) and evaluates the pre-trained BGE-Base model across multiple dimensions. This serves as the control group to measure the lift provided by Matryoshka fine-tuning later.

In [1]:
import sys, os, yaml, shutil, torch, pickle
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Path Setup
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
from src.esci.data import sample_dataset
from src.esci.system_a import encode_systemA
from src.esci.system_b import build_candidates
from src.esci.artifacts import load_artifacts
from src.esci.sparse_retrievers import SPLADEFast, BM25Fast
from src.esci.metrics import compute_recall_metrics, compute_ndcg_metrics


# Load Config
with open(project_root / "configs" / "esci.yaml") as f:
    cfg = yaml.safe_load(f)

# Load Data
print("Loading Processed Data...")
pair_df = pd.read_parquet(cfg["paths"]["processed_dir"] + "/pair_df.parquet")
print("Processed Data is LOADED")

Loading Processed Data...
Processed Data is LOADED


### Ground Truth Construction

In [2]:
# We pre-calculate the 'q_rels' (Query Relevance) map.
# This dictionary maps query_id -> [list of grades].
# It acts as the immutable standard against which all retrieval strategies 
# (Dense, Sparse, Hybrid) will be measured using nDCG and Recall.

print("📊 Building Ground Truth Map (q_rels)...")
q_rels = pair_df.groupby("query_id")["grade"].apply(list).to_dict()
print("DONE")

📊 Building Ground Truth Map (q_rels)...
DONE


### Clear Aartifacts & Encode

To save storage space and avoid confusion between artifacts from baseline model vs finetuned model, we clear the artifacts folder once we start encoding with the baseline model.

In [3]:
artifact_dir = Path(cfg["paths"]["artifacts_dir"])

if os.path.exists(artifact_dir):
    print(f"🧹 Clearing old artifacts in {artifact_dir}...")
    shutil.rmtree(artifact_dir)
    os.makedirs(artifact_dir)

base_model = cfg["biencoder_model"]
print(f"🚀 Encoding with Baseline Model: {base_model}")
encode_systemA(pair_df, cfg, model_override=base_model)

🧹 Clearing old artifacts in ..\artifacts\systemA...
🚀 Encoding with Baseline Model: BAAI/bge-base-en-v1.5
🚀 Encoding with System A model: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📏 Max Seq Length: 256
    -> Preparing Product List...
    -> Encoding 164001 Products...


Batches:   0%|          | 0/5126 [00:00<?, ?it/s]

    -> Preparing Query List...
    👉 Applying BGE instruction prefix: 'Represent this sentence for searching relevant passages: '
    -> Encoding 8908 Queries...


Batches:   0%|          | 0/279 [00:00<?, ?it/s]

✅ System A Encoding Complete.


### Build Sparse Indices

In [4]:
print("⚡ Pre-building Sparse Indices...")
_, _, prod_df, _, _ = load_artifacts("us", "test", cfg)

# Build SPLADE
splade = SPLADEFast(
    cfg["sparse"]["splade_model"],
    batch_size=cfg["sparse"]["batch_size"]
)

splade.build_index(
    prod_df["product_text_dense"].tolist(),
    prod_df["product_id"].tolist()
)

print("✅ SPLADE built:", splade.doc_matrix.shape)

⚡ Pre-building Sparse Indices...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️ Building Fast SPLADE index for 164001 docs...
✅ SPLADE index built. Matrix shape: (164001, 30522)
✅ SPLADE built: torch.Size([164001, 30522])


In [5]:
# Build BM25
bm25 = BM25Fast()
bm25.build_index(
    texts=prod_df["product_title"].fillna("").astype(str).tolist(),
    pids=prod_df["product_id"].tolist()
)

🏗️ Building Fast GPU BM25 Index (stemming + normalize) for 164001 docs...
✅ BM25 index built. Matrix shape: (164001, 96275)
✅ BM25 vocab size: 96275


### Saving artifacts

In [6]:
save_dir = "../artifacts/systemA"
os.makedirs(save_dir, exist_ok=True)

In [7]:
splade_path = f"{save_dir}/splade_data.pt"
torch.save({
    "doc_matrix": splade.doc_matrix,
    "pid_list": splade.pid_list,
    "model_state_dict": splade.model.state_dict()
}, splade_path)

print(f"✅ SPLADE saved to: {splade_path}")

✅ SPLADE saved to: ../artifacts/systemA/splade_data.pt


In [8]:
bm25_path = f"{save_dir}/bm25_retriever.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)

print(f"✅ BM25 saved to: {bm25_path}")

✅ BM25 saved to: ../artifacts/systemA/bm25_retriever.pkl


### Loading Artifacts

In [3]:
# Loading SPLADE/BM25 from a .pt checkpoint is ~10x faster than rebuilding it.
# We explicitly map the sparse matrix to the GPU ('cuda') to ensure
# that subsequent scoring operations utilize Tensor Cores.

print("⚡ Loading SPLADE Matrix...")
splade_data = torch.load("../artifacts/systemA/splade_data.pt", map_location="cpu")

splade_loaded = SPLADEFast(
    cfg["sparse"]["splade_model"], 
    batch_size=cfg["sparse"]["batch_size"]
)

splade_loaded.doc_matrix = splade_data["doc_matrix"]
splade_loaded.pid_list   = splade_data["pid_list"]

# 🔴 CRITICAL: device alignment
splade_loaded.doc_matrix = splade_loaded.doc_matrix.to(splade_loaded.device)

# Safety
assert splade_loaded.doc_matrix.shape[0] == len(splade_loaded.pid_list)

print("⚡ Loading BM25 Index...")
with open("../artifacts/systemA/bm25_retriever.pkl", "rb") as f:
    bm25_loaded = pickle.load(f)

splade = splade_loaded
bm25 = bm25_loaded

⚡ Loading SPLADE Matrix...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Loading BM25 Index...


## Running Experiment Loop

In [4]:
results = []
dims = cfg["matryoshka"]["dims"]
strategies = {
    "Dense Only": ["dense"],
    "Dense + BM25": ["dense", "bm25"],
    "Dense + SPLADE": ["dense", "splade"],
    "Dense + BM25 + SPLADE": ["dense", "bm25", "splade"]
}

max_len = cfg["matryoshka"]["max_seq_length"]

for strat, sources in strategies.items():
    print(f"\n🔵 Strategy: {strat}")
    cfg["retrieval"]["sources"] = sources
    
    for dim in dims:
        df, qps = build_candidates(
            cfg, split="test", override_dim=dim,
            prebuilt_splade=splade if "splade" in sources else None,
            prebuilt_bm25=bm25 if "bm25" in sources else None
        )
        
        # Pass q_rels here!
        # Compute them separately
        recall_results = compute_recall_metrics(df, q_rels)
        ndcg_results = compute_ndcg_metrics(df, q_rels)
        
        # Merge them into one final dictionary
        final_metrics = {**recall_results, **ndcg_results}
        
        results.append({
            "Model": "Baseline",
            "Strategy": strat,
            "Dimension": dim,
            "Max Length": max_len,
            "Recall@50": final_metrics["Recall@50"],
            "Recall@100": final_metrics["Recall@100"],
            "Recall@200": final_metrics["Recall@200"],
            "nDCG@10": final_metrics["nDCG@10"],
            "nDCG@20": final_metrics["nDCG@20"],
            "nDCG@50": final_metrics["nDCG@50"],
            "QPS": round(qps, 2)
        })


🔵 Strategy: Dense Only
⚡ Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

🔵 Strategy: Dense + BM25
⚡ Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

🔵 Strategy: Dense + SPLADE
⚡ Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
⚡ Using FAISS 

In [5]:
# Save & Display
output_dir = "../results"
if not os.path.exists(output_dir): os.makedirs(output_dir)
results_df = pd.DataFrame(results)
results_df.to_csv(f"{output_dir}/baseline_metrics_per_dim.csv", index=False)

In [6]:
print("\n🏆 QPS & Performance by Dimension:")
display(results_df.pivot(index="Dimension", columns="Strategy", values=["Recall@200", "QPS"]))


🏆 QPS & Performance by Dimension:


Recall@200                                                  \
Strategy  Dense + BM25 Dense + BM25 + SPLADE Dense + SPLADE Dense Only   
Dimension                                                                
64            0.483983              0.650744       0.572161   0.426956   
128           0.630038              0.706292       0.668213   0.588258   
256           0.708182              0.745571       0.726408   0.681114   
512           0.744027              0.766591       0.753579   0.724900   
768           0.756639              0.774405       0.764270   0.739768   

                   QPS                                                  
Strategy  Dense + BM25 Dense + BM25 + SPLADE Dense + SPLADE Dense Only  
Dimension                                                               
64              144.04                 70.80          80.18     329.28  
128             101.94                 59.99          67.91     204.05  
256              68.66                 43.42          46.60     108.34  
512              41.81                 26.67          29.82      56.52  
768              28.47                 19.24          22.48      36.19

In [7]:
results_df

,Model,Strategy,Dimension,Max Length,Recall@50,Recall@100,Recall@200,nDCG@10,nDCG@20,nDCG@50,QPS
0,Baseline,Dense Only,768,256,0.569735,0.662507,0.739768,0.433009,0.463072,0.515612,36.19
1,Baseline,Dense Only,512,256,0.555104,0.647921,0.724900,0.422693,0.451474,0.503189,56.52
2,Baseline,Dense Only,256,256,0.511757,0.601916,0.681114,0.395021,0.420819,0.469609,108.34
3,Baseline,Dense Only,128,256,0.430857,0.512219,0.588258,0.338874,0.360226,0.403421,204.05
4,Baseline,Dense Only,64,256,0.290365,0.357380,0.426956,0.235852,0.249192,0.281322,329.28
5,Baseline,Dense + BM25,768,256,0.586373,0.679313,0.756639,0.447023,0.478648,0.530854,28.47
6,Baseline,Dense + BM25,512,256,0.573561,0.667018,0.744027,0.439004,0.469240,0.520690,41.81
7,Baseline,Dense + BM25,256,256,0.535216,0.628261,0.708182,0.419462,0.445571,0.494243,68.66
8,Baseline,Dense + BM25,128,256,0.461280,0.547308,0.630038,0.377115,0.396357,0.439051,101.94
9,Baseline,Dense + BM25,64,256,0.326091,0.401858,0.483983,0.291657,0.299827,0.332209,144.04
